<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 14.OpenCV 摄像头AprilTag标识识别抓取实验

@－－－－DICE－－－－<br>
@  文件名：14.OpenCV 摄像头AprilTag识别抓取实验.py <br>
@  版本：V2.0 <br>
@  说明：我们第一步需要开启摄像头作为显示
我们来学习一下摄像头的开启方式之一吧，一般来说，对于我们使用摄像头的时候，首先我们需要测试我们的摄像头是否正常工作，画面是否清晰，让我们一起来测试我们的摄像头吧！！！<br>

我们开始测试摄像头是否可用

### Opencv Apriltag识别（python）

In [1]:
#!/usr/bin/python3
# coding=utf8
import sys
import cv2
import math
import time
import threading
import numpy as np
import apriltag
# 自定义库
sys.path.append("./")
from camera import Camera
#from color_block_detector import ColorBlockDetector
from image2workspace import Image2Workspace
from arm5dof_uservo import Arm5DoFUServo
from arm_application import ArmApplication

### 创建显示交互界面

In [2]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg


frame_img = widgets.Image(format='jpeg', width=640, height=480)
display(frame_img)

Image(value=b'', format='jpeg', height='480', width='640')

In [3]:
# 线程函数操作库
import threading # 线程
import ctypes
import inspect

# 线程结束代码
def _async_raise(tid, exctype):
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        raise SystemError("PyThreadState_SetAsyncExc failed")
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)

### Apriltag识别

In [4]:
#####################################
## 物块抓取的配置文件
#####################################
# - 物块像素尺寸最小值
cubic_min_size = 100
# - 物块像素尺寸最大值
cubic_max_size = 300
# - 机械臂抓取位姿偏移量
arm_offset_xy = -10.0 # XY平面上的偏移量
arm_offset_z = 0.0 # Z轴上的偏移量
# - 抓取阈值
# 	在某个位置检测到多少次就进行抓取
cubic_count_threshold = 20
# - 是否只抓取工作区内的物块
# 	True: 只抓取工作区内的物块
#   False: 工作区外的也抓
grab_cubic_in_ws = False
#####################################

object_center_x = 0.0
object_center_y = 0.0

# 创建机械臂
arm_config_folder = "config"
arm = Arm5DoFUServo(config_folder=arm_config_folder)
# 运动到图像拍摄点
arm.set_tool_pose(pose_name="capture_image")
# 创建机械臂APP
arm_app = ArmApplication(arm, config_folder=arm_config_folder)
# 要识别的Apriltag 标签列表
Apriltag_name_list = ["0","1","2","3","4"]
# 创建摄像头
camera = Camera()
# 初始相机
capture = camera.capture
# 载入标定数据
camera.load_cam_calib_data()
# 色块识别器
#color_block_detector = ColorBlockDetector()
# 像素到工作台坐标映射器
img2ws = Image2Workspace(camera)


def candi_workpiece_encoding(wx, wy, delta=20):
    '''工件坐标编码, 坐标单位mm'''
    return int(wx/delta) + int(wy/delta)*1000

def justify_workspace_posi(wx, wy, wz):
    '''调整工作空间坐标'''
    theta = math.atan2(wy, wx)
    r = math.sqrt(wx*wx + wy*wy)
    new_r = r + arm_offset_xy
    new_wx = new_r * math.cos(theta)
    new_wy = new_r * math.sin(theta) 
    new_wz = wz + arm_offset_z
    return new_wx, new_wy, new_wz

# 检测apriltag
detector = apriltag.Detector(searchpath=apriltag._get_demo_searchpath())

def apriltagDetect(img):
    global object_center_x, object_center_y
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    detections = detector.detect(gray, return_image=False)

    if len(detections) != 0:
        for detection in detections:                       
            corners = np.rint(detection.corners)  # 获取四个角点
            cv2.drawContours(img, [np.array(corners, np.int)], -1, (0, 255, 255), 2)
            tag_family = str(detection.tag_family, encoding='utf-8')  # 获取tag_family
            tag_id = int(detection.tag_id)  # 获取tag_id
            object_center_x, object_center_y = int(detection.center[0]), int(detection.center[1])  # 中心点
            object_angle = int(math.degrees(math.atan2(corners[0][1] - corners[1][1], corners[0][0] - corners[1][0])))  # 计算旋转角
            return tag_family, tag_id
    return None, None

def Video_display():
    global state
    global tag_id
    global action_finish
    global object_center_x, object_center_y
    # 机械臂物块抓取
    # Key: 位置编码
    # Value: (颜色名称, 计数)
    candi_workpiece_dict = {} # 候选工件字典
    while True:
        ret, img = camera.read()
        if ret:
            frame = img.copy()
            img_h, img_w = img.shape[:2]
            tag_family, tag_id = apriltagDetect(img) # apriltag检测
            if tag_id is not None: 
                # 打印Apritag标签的中心坐标
                # print('X:',object_center_x,'Y:',object_center_y)
                
                # 将图像坐标系转换为工作台坐标系
                wx, wy = img2ws.img2workspace(object_center_x, object_center_y)
                # 判断是否在工作区内
                if grab_cubic_in_ws:
                    if abs(wx) > img2ws.x0 or abs(wy) > img2ws.y0:
                        # 在工作区格子外，不处理
                        print(f"超出工作区范围: wx = {wx} wy = {wy}")
                        continue 
                # 对位置进行编码
                code = candi_workpiece_encoding(wx, wy)
                if code not in candi_workpiece_dict:
                    candi_workpiece_dict[code] = [tag_id, 1]
                else:
                    candi_workpiece_dict[code][0] = tag_id
                    candi_workpiece_dict[code][1] += 1
                print(f"纳入统计: wx = {wx} wy = {wy} 编码 = {code}")
                if(candi_workpiece_dict[code][1] >= cubic_count_threshold):
                    # 抓取工件
                    print("- 抓取工件")
                    print(f"Apriltag物块ID {tag_id} wx={wx:.1f} wy={wy:.1f} mm")
                    wx2, wy2, wz2 = justify_workspace_posi(wx, wy, img2ws.object_height)
                    arm_app.grab_cubic(wx2-15, wy2+10, wz2,str(tag_id))
                    # 清空投票字典
                    candi_workpiece_dict = {}
                    #break
                # 显示先关信息
                cv2.putText(img, "tag_id: " + str(tag_id), (10, img.shape[0] - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, [0, 255, 255], 2)
                cv2.putText(img, "tag_family: " + tag_family, (10, img.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, [0, 255, 255], 2)
            else:
                cv2.putText(img, "tag_id: None", (10, img.shape[0] - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, [0, 255, 255], 2)
                cv2.putText(img, "tag_family: None", (10, img.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, [0, 255, 255], 2)    
                
            frame_img.value = bgr8_to_jpeg(img)
            key = cv2.waitKey(1)
            if key == 27:
                break
        else:
            time.sleep(0.01)
    cv2.destroyAllWindows()

In [5]:
# 开始线程
t = threading.Thread(target=Video_display)
t.setDaemon(True)
t.start()

/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-i

纳入统计: wx = -0.9325500594236189 wy = -21.791427620026056 编码 = -1000
纳入统计: wx = 38.11935345460299 wy = -29.319751644312973 编码 = -999
纳入统计: wx = 36.99603393667026 wy = -28.590199513235376 编码 = -999
纳入统计: wx = 35.84628242168172 wy = -28.392230410845514 编码 = -999
纳入统计: wx = 35.46303191668554 wy = -28.326240710048893 编码 = -999
纳入统计: wx = 35.46303191668554 wy = -28.326240710048893 编码 = -999
纳入统计: wx = 35.066565413161456 wy = -28.526042523596132 编码 = -999
纳入统计: wx = 34.670098909637375 wy = -28.725844337143386 编码 = -999
纳入统计: wx = 34.670098909637375 wy = -28.725844337143386 编码 = -999
纳入统计: wx = 34.670098909637375 wy = -28.725844337143386 编码 = -999
纳入统计: wx = 34.28684840464119 wy = -28.659854636346765 编码 = -999
纳入统计: wx = 33.13709688965265 wy = -28.461885533956902 编码 = -999
纳入统计: wx = 32.3573798811324 wy = -28.59569764670752 编码 = -999
纳入统计: wx = 31.96091337760832 wy = -28.79549946025476 编码 = -999
纳入统计: wx = 31.16798037056016 wy = -29.19510308734924 编码 = -999


/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(


纳入统计: wx = 28.445578840003208 wy = -29.530549724804487 编码 = -999
纳入统计: wx = 26.516110316494405 wy = -29.466392735165243 编码 = -999
纳入统计: wx = 26.13285981149822 wy = -29.400403034368622 编码 = -999
纳入统计: wx = 25.35314280297797 wy = -29.53421514711924 编码 = -999
纳入统计: wx = 25.736393307974154 wy = -29.60020484791586 编码 = -999
纳入统计: wx = 25.736393307974154 wy = -29.60020484791586 编码 = -999
- 抓取工件
Apriltag物块ID 0 wx=25.7 wy=-29.6 mm
纳入统计: wx = 25.736393307974154 wy = -29.60020484791586 编码 = -999
纳入统计: wx = 25.35314280297797 wy = -29.53421514711924 编码 = -999
纳入统计: wx = 100.62879340762524 wy = -62.60052918143643 编码 = -2995
纳入统计: wx = 97.86677078611041 wy = -48.185463128386004 编码 = -1996
纳入统计: wx = 82.48388659215162 wy = -46.60904115389663 编码 = -1996
纳入统计: wx = 96.69058727406606 wy = -48.519077054683876 编码 = -1996
纳入统计: wx = 97.07383777906225 wy = -48.5850667554805 编码 = -1996
纳入统计: wx = 96.67737127553816 wy = -48.784868569027736 编码 = -1996
纳入统计: wx = 96.66415527701027 wy = -49.050660083371596 编码 = 

/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(


纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -4.791554368005777 wy = -60.53283172459069 编码 = -3000
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -4.804770366533674 wy = -60.79862323893454 编码 = -3000
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -4.791554368005777 wy = -60.53283172459069 编码 = -3000
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -4.791554368005777 wy = -60.53283172459069 编码 = -3000
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -4.791554368005777 wy = -60.53283172459069 编码 = -3000
- 抓取工件
Apriltag物块ID 1 wx=-4.8 wy=-60.5 mm
纳入统计: wx = -5.174804873001946 wy = -60.46684202379407 编码 = -3000
纳入统计: wx = -4.804770366533674 wy = -60.79862323893454 编码 = -3000
纳入统计: wx = -12.905868060939227 wy = -44.92811834

... (7 outputs left out)

纳入统计: wx = 31.075481833177776 wy = -23.281700070987654 编码 = -999
纳入统计: wx = 30.322196821713305 wy = -22.883929155050552 编码 = -999
纳入统计: wx = 29.952162315245033 wy = -22.55214793991007 编码 = -999
纳入统计: wx = 29.198877303780577 wy = -22.15437702397297 编码 = -999
纳入统计: wx = 29.185661305252665 wy = -22.42016853831683 编码 = -999
纳入统计: wx = 28.7891948017286 wy = -22.61997035186407 编码 = -999
纳入统计: wx = 28.7891948017286 wy = -22.61997035186407 编码 = -999
纳入统计: wx = 28.7891948017286 wy = -22.61997035186407 编码 = -999
纳入统计: wx = 28.7891948017286 wy = -22.61997035186407 编码 = -999
纳入统计: wx = 28.405944296732415 wy = -22.553980651067448 编码 = -999
纳入统计: wx = 28.419160295260312 wy = -22.288189136723588 编码 = -999
纳入统计: wx = 28.432376293788224 wy = -22.022397622379728 编码 = -999
纳入统计: wx = 28.419160295260312 wy = -22.288189136723588 编码 = -999
纳入统计: wx = 28.02269379173623 wy = -22.487990950270827 编码 = -999
纳入统计: wx = 28.02269379173623 wy = -22.487990950270827 编码 = -999
纳入统计: wx = 28.405944296732415 wy = -22.553

/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(


纳入统计: wx = 51.480243683044776 wy = -40.46650084633876 编码 = -1998
纳入统计: wx = 51.17628916921599 wy = -38.80576205947898 编码 = -998
纳入统计: wx = 51.62561966685166 wy = -37.542794188556286 编码 = -998
纳入统计: wx = 52.07495016448735 wy = -36.27982631763359 编码 = -998
纳入统计: wx = 51.40096114419029 wy = -34.287306315633316 编码 = -998
纳入统计: wx = 51.1102226288894 wy = -32.36077601442966 编码 = -998
纳入统计: wx = 20.199091709477855 wy = -24.35769510646476 编码 = -999
纳入统计: wx = 20.22552370653365 wy = -23.826112077777026 编码 = -999
纳入统计: wx = 21.401707218577982 wy = -23.492498151479168 编码 = -999
纳入统计: wx = 22.16820822857035 wy = -23.62447755307241 编码 = -999
纳入统计: wx = 22.55145873356652 wy = -23.69046725386903 编码 = -999
纳入统计: wx = 22.921493240034806 wy = -24.02224846900951 编码 = -999
纳入统计: wx = 23.701210248555057 wy = -23.888436356258893 编码 = -999
纳入统计: wx = 24.49414325560322 wy = -23.488832729164415 编码 = -999
纳入统计: wx = 25.657110769119654 wy = -23.421010317210417 编码 = -999
纳入统计: wx = 25.657110769119654 wy = -23.421

/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-install-hybbb0ti/apriltag/core/contrib/apriltag_quad_contour.c:467: warning: count < 8 :(
/tmp/pip-i

In [6]:
# 结束线程
stop_thread(t)